# Sparx remote model (Kaggle)
Runs Ollama on a free Kaggle GPU and exposes it through a token-protected Cloudflare tunnel.

Before running: **Settings -> Accelerator: GPU T4 x2**, **Internet: On**.
Optional: add a Kaggle Secret named `SPARX_TOKEN` (otherwise a random token is generated and printed).
Run the cells in order and leave the last one running. The session ends after a few hours; rerun to get a new URL.

In [ ]:
MODEL = "qwen2.5:14b"      # fits T4 x2 (32 GB). Try "qwen2.5:32b" for more quality, slower.
PORT_OLLAMA, PORT_PROXY = 11434, 8000

In [ ]:
%%bash
apt-get install -y zstd >/dev/null 2>&1
curl -fsSL https://ollama.com/install.sh | sh
curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
chmod +x /usr/local/bin/cloudflared
pip install -q fastapi uvicorn httpx

In [ ]:
import os, subprocess, time, secrets, httpx
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("SPARX_TOKEN")
except Exception:
    TOKEN = secrets.token_urlsafe(24)

subprocess.Popen(["ollama", "serve"], env={**os.environ, "OLLAMA_HOST": f"127.0.0.1:{PORT_OLLAMA}"},
                 stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        httpx.get(f"http://127.0.0.1:{PORT_OLLAMA}/api/tags").raise_for_status(); break
    except Exception:
        time.sleep(1)
subprocess.run(["ollama", "pull", MODEL], check=True)

In [ ]:
# Proxy: only requests with the bearer token reach Ollama.
open("proxy.py", "w").write(f"""
import httpx
from fastapi import FastAPI, Request, Response
from fastapi.responses import StreamingResponse
app, client = FastAPI(), httpx.AsyncClient(base_url="http://127.0.0.1:{PORT_OLLAMA}", timeout=600)

@app.api_route("/{{path:path}}", methods=["GET", "POST"])
async def forward(path: str, request: Request):
    if request.headers.get("authorization") != "Bearer {TOKEN}":
        return Response("unauthorized", status_code=401)
    req = client.build_request(request.method, "/" + path, content=await request.body(),
                               headers={{"content-type": request.headers.get("content-type", "application/json")}})
    r = await client.send(req, stream=True)
    return StreamingResponse(r.aiter_raw(), status_code=r.status_code, background=None,
                             media_type=r.headers.get("content-type"))
""")
subprocess.Popen(["python", "-m", "uvicorn", "proxy:app", "--port", str(PORT_PROXY)], stdout=open("proxy.log", "w"), stderr=subprocess.STDOUT)
time.sleep(3)

In [ ]:
import re, json
tun = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT_PROXY}"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0); break

# Publish URL + token to a secret gist so `sparx kaggle-sync --watch` updates ~/.sparx/.env by itself.
# Needs Kaggle Secrets GITHUB_TOKEN (gist scope) and SPARX_GIST_ID (id of a secret gist containing sparx_server.json).
try:
    from kaggle_secrets import UserSecretsClient
    gh, gist = UserSecretsClient().get_secret("GITHUB_TOKEN"), UserSecretsClient().get_secret("SPARX_GIST_ID")
    body = json.dumps({"files": {"sparx_server.json": {"content": json.dumps({"base_url": f"{url}/v1", "token": TOKEN})}}})
    r = httpx.patch(f"https://api.github.com/gists/{gist}", content=body,
                    headers={"Authorization": f"Bearer {gh}", "Accept": "application/vnd.github+json"})
    r.raise_for_status()
    print("Published to gist; your PC picks it up with: sparx kaggle-sync --watch")
except Exception as e:
    print(f"(not auto-published: {e})")

print("\nManual fallback - put these in ~/.sparx/.env (then restart sparx):\n")
print(f"BROWSERPILOT_BASE_URL={url}/v1")
print(f"BROWSERPILOT_API_KEY={TOKEN}")
print(f"\nThen: sparx config models.default '\"custom/{MODEL}\"'   (or add it to models.fallbacks)")

In [ ]:
# Keep the session alive and show requests; stop this cell to shut the server down.
while True:
    time.sleep(60)